In [30]:
import numpy as np
import random
import itertools
import pandas as pd
import os
import ot
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import glob 
from copy import copy
from scipy.sparse import csr_matrix
from scipy.stats import pareto
import scipy.spatial.distance as ssd
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.cluster import KMeans
from sklearn_extra.cluster import KMedoids
from sklearn.decomposition import NMF
from sklearn.metrics.cluster import normalized_mutual_info_score as NMI
from EDRep_main.EDRep import NodeEmbedding  
from tqdm.notebook import tqdm



In [2]:
def generateSequence(outputfolder, model, args, n_graphs, verbose = True, append_name = '.csv'):
    '''This function generates a sequence of graphs that have a common generative model in each block
    
    Use: generateSequence(outputfolder, model, args, n_graphs)
    
    Inputs: 
        * outputfolder : directory to which the files will be saved
        * model (function): function of the generative model to be considered
        * args (list of lists): arguments of model
        * n_graphs (int): number of graphs to be generated

    Optional inputs:
        * verbose (boolean): if True (default) prints the progress bar
        * append_name (string): what goes at the end of the file name. Can be used to change the extension or also part of the name of the file. By default `.csv`
        
    Outputs:
        * this function will output the edgelist representation of the generated graphs numbered in ascending order.       
    '''
    
    for i in range(n_graphs):
            
        df = model(args)
        df.to_csv(outputfolder + '/EL' + str(i+1) + append_name, index = False)

        # print progress bar
        if verbose == 1:
            print("[%-25s] %d%%" % ('='*(int(i/(n_graphs)*25)) + '>', (i+1)/(n_graphs)*100), end = '\r')
        
    return

In [3]:
def DCSBM(args):
    ''' 
    Function that generates a graph from the degree-corrected stochastic block model with
    n nodes and k communities
    
    Note that, with an appropriate choice of the parameters, this function can be used to
    generate an SBM or and ER random graph
    
    Use:
        edge_list = DCSBM(C,c, ℓ, θ, symmetric, make_connected)
    Inputs:
        args = C, c, ℓ, θ
            * C (array of size k x k) : affinity matrix of the network C
            * c (scalar) : average degree of the network
            * ℓ (array of size n) : vector containing the label of each node
            * θ  (array of size n) : vector with the intrinsic probability connection of each node
            * symmetric (bool): if True it returns an undirected graph
            * make_connected (bool): if True forces the graph to be connected adding edges
    
    Outputs:
        * edge_list (pandas dataframe) : edge list representation of the graph (ij)
        
    '''
    
    C, c, ℓ, θ, symmetric, make_connected  = args

    # number of communities
    k = len(np.unique(ℓ))
    
    # number of nodes
    n = len(θ)
    
    # (k x n) matrix where we store the value of the affinity wrt a given label for each node
    c_v = C[ℓ].T
    
    fs = list()
    ss = list()

    # we choose the nodes that should get connected wp = θ_i/n
    if symmetric:
        first = np.random.choice(n,int(n*c/2),p = θ/n)

    else:  
        first = np.random.choice(n,int(n*c),p = θ/n) 

    for i in range(k): 
        v = θ*c_v[i]
        
        # among the nodes of first, select those with label i
        first_selected = first[ℓ[first] == i]
        fs.append(first_selected.tolist())
        
        # choose the nodes to connect to the first_selected
        second_selected = np.random.choice(n,len(first_selected), p = v/np.sum(v))
        ss.append(second_selected.tolist())

    fs = list(itertools.chain(*fs))
    ss = list(itertools.chain(*ss))

    fs = np.array(fs)
    ss  = np.array(ss)


    # add nodes not appearing in the vector first, to ensure the graph is connected
    if make_connected:
        idx = np.arange(n)[np.logical_not(np.isin(np.arange(n), fs))]
        fs = np.concatenate([fs, idx])
        ss = np.concatenate([ss, np.argmax(θ)*np.ones(len(idx))])
    

    # create the edge list from the connection defined earlier
    edge_list = np.column_stack((fs,ss)) 
    
    # remove edges appearing more then once
    edge_list = np.unique(edge_list, axis = 0) 

    # replace self edges
    idx = edge_list[:,0] == edge_list[:,1]
    edge_list[:,1][idx] += 1
    idx = edge_list[:,1] == n
    edge_list[:,1][idx] = 0

    if symmetric:
        df = pd.DataFrame(columns = ['i', 'j'])
        M = np.maximum(edge_list[:,0], edge_list[:,1])
        m = np.minimum(edge_list[:,0], edge_list[:,1])
        df.i = np.concatenate([M, m])
        df.j = np.concatenate([m, M])

    else:
        df = pd.DataFrame(columns = ['i', 'j'])
        df.i = edge_list[:,0]
        df.j = edge_list[:,1]

    # add an edge to nodes with degree 0
    A = csr_matrix((np.ones(len(df)), (df.i, df.j)), shape = (n,n))
    d = A@np.ones(n)
    idx = d == 0


    return df


In [4]:
def GeometricModel(args):
    '''Generates an instance of a modified Geometric model from a given configuration of the points in space
    
    Use: A = GeometricModel(X, d, β)
    
    Inputs: args
        * X (array): coordinates of the points used to form edges
        * d (float): average degree
        * β (float): noise parameter
        
    Ouptput:
        * A (sparse csr_matrix): adjacency matrix of the generated graph'''


    X, d, β = args
    n = np.shape(X)[0]

    # compute the pairwise distance matrix
    no = (X**2)@np.ones(2)
    D = np.zeros((n,n))
    for i in range(n):
        D[i] += no
        D[:,i] += no

    D -= 2*X@X.T
    D = np.sqrt(np.abs(D))

    # compute the probability matrix
    P = np.exp(-β*D)
    P = P - np.diag(np.diag(P))
    p = P@np.ones(n)
    P = np.diag(p**(-1)).dot(P)
    
    # draw the edges
    idx1 = np.concatenate([np.ones(int(d/2))*i for i in range(n)])
    idx2 = np.concatenate([np.random.choice(np.arange(n), int(d/2), p = P[i], replace = False) for i in range(n)])

    df = pd.DataFrame(columns = ['i', 'j'])

    # symmetrize
    df.i = np.concatenate([idx1, idx2])
    df.j = np.concatenate([idx2, idx1])

    return df


In [5]:
def EmbDistance(X, Y, distance_type = 'unmatched'):
    '''This function computes the distance between 

    Use: d = EmbDistance(X, Y)

    Inputs:
        * X, Y (arrays): input embeddings corresponding to the two temporal graphs. The number of rows of the two matrices must be the same.
        
    Optional inputs:
        * distance_type (string): can be 'unmatched' or 'matched'

    Output:
        * d (float): distance between the two graphs.
    '''

    n1, d1 = X.shape
    n2, d2 = Y.shape

    # run initial checks

    if d1 != d2:
        raise DeprecationWarning('The embedding matrices have different dimensions')
    else:
        d = d1

    
    if distance_type not in ['unmatched', 'matched']:
        raise DeprecationWarning('The distance type is not valid')
    
    else:
        if (distance_type == 'matched') and (n1 != n2):
            raise DeprecationWarning("The input matrices do not have the same size")
        else:
            n = n1

    if distance_type == 'matched':
        Mxx = X.T@X
        Mxy = X.T@Y
        Myy = Y.T@Y
        d = np.sqrt(np.abs(np.linalg.norm(Mxx)**2 + np.linalg.norm(Myy)**2 - 2*np.linalg.norm(Mxy)**2))

    else:
        n1, n2 = X.shape[0], Y.shape[0]
        λ1 = np.linalg.eigvalsh(X.T@X)/n1
        λ2 = np.linalg.eigvalsh(Y.T@Y)/n2
        d = np.linalg.norm(λ1-λ2)
  
    return d

In [6]:
def get_adj(filepath):
    df = pd.read_csv(filepath)
    G = nx.from_pandas_edgelist(df, 'i', 'j')
    A = nx.adjacency_matrix(G).astype(float)

    return A

In [ ]:
def node_embedding(g, dim_embedding,seed=42):
    rdn_state = np.random.get_state()
    random.seed(seed)
    np.random.seed(seed)
    embedding_a = NodeEmbedding(g, dim=dim_embedding, k=1, verbose=False)
    X_total = embedding_a.X
    np.random.set_state(rdn_state)
    return X_total


In [8]:
def w2_distance(emb_1, emb_2):
    S_1=emb_1@emb_1.T
    S_2=emb_2@emb_2.T
    vals_S_1=S_1.flatten()
    vals_S_2=S_2.flatten()
    wd2 = ot.wasserstein_1d(vals_S_1, vals_S_2, p=2)**(1/2)
    return wd2

In [9]:
def create_embeddings(path_files, dim):
    embeddings = []

    for file in tqdm(path_files, desc="Calcolando embeddings"):
        a = get_adj(file)
        emb = node_embedding(a, dim)
        embeddings.append(emb)
        
    return embeddings


In [ ]:
def distances(embeddings, n):
    old_d = []
    new_d = []
    subset = embeddings[:n]
    
    k = len(subset)

    total_iters = k * (k - 1) // 2

    for a, b in tqdm(itertools.combinations(subset, 2), total=total_iters, desc="Calcolando distanze"):
        old = EmbDistance(a, b, 'unmatched')
        new = w2_distance(a, b)
        old_d.append(old)
        new_d.append(new)
 
    return old_d, new_d


In [12]:
def comparison_plot(old_d, new_d, name):
    plt.figure(figsize=(8, 6))

    plt.scatter(old_d, new_d,alpha=0.6)

    plt.xlabel("unmatched")
    plt.ylabel("wasserstein")
    plt.title(f"Comparing distances for {name}")
    plt.grid(True)
    plt.show()

In [13]:
output_folder_dcsbm = r"C:\Users\Utente\Desktop\progetto vscode\data\synthetic_graphs\generated_dcsbm"
os.makedirs(output_folder_dcsbm, exist_ok=True)

output_folder_er= r"C:\Users\Utente\Desktop\progetto vscode\data\synthetic_graphs\generated_er"
os.makedirs(output_folder_er, exist_ok=True)

output_folder_conf= r"C:\Users\Utente\Desktop\progetto vscode\data\synthetic_graphs\generated_configuration"
os.makedirs(output_folder_conf, exist_ok=True)

output_folder_geo= r"C:\Users\Utente\Desktop\progetto vscode\data\synthetic_graphs\generated_geo"
os.makedirs(output_folder_geo, exist_ok=True)

In [31]:

#PARAMETRI CONDIVISI 
n_nodes = 1000
γ = 0.8 
symmetric = True
make_connected = True
c = 10 

Calcolo c_in e c_out mantenendo c e k costanti, al variare di alpha.
   

In [26]:
def compute_cin_cout(c, k, alpha):
    c_out = c - alpha * np.sqrt(c)
    c_in = c + (k - 1) * alpha * np.sqrt(c)
    
    return c_in, c_out


In [27]:
c_cost = 10 
k_cost = 4    

# range di variazione di alpha
alpha_array = np.linspace(0, 2, 5)

c_in_array, c_out_array = compute_cin_cout(c_cost, k_cost, alpha_array)
print(c_in_array)
df_results = pd.DataFrame({
    'alpha': alpha_array,
    'c_in': c_in_array,
    'c_out': c_out_array
})

print(f"c = {c_cost}, k = {k_cost}\n")
print(df_results.to_string(index=False))


[10.         14.74341649 19.48683298 24.23024947 28.97366596]
c = 10, k = 4

 alpha      c_in     c_out
   0.0 10.000000 10.000000
   0.5 14.743416  8.418861
   1.0 19.486833  6.837722
   1.5 24.230249  5.256584
   2.0 28.973666  3.675445


DCSBM

In [32]:
k = 4    

alpha_array = np.linspace(0, 2, 5)

c_in_a, c_out_a = compute_cin_cout(c, k, alpha_array)

C_dcsbm = np.ones((k,k)) * c_out_a
C_dcsbm += np.diag(np.ones(k)) * (c_in_a - c_out_a)

symmetric = True
make_connected = True

θ_dcsbm = np.ones(n_nodes)
ℓ_dcsbm = np.zeros(n_nodes)
for i in range(k):
    ℓ_dcsbm[i*int(n_nodes/k): (i+1)*int(n_nodes/k)] = i
ℓ_dcsbm = ℓ_dcsbm.astype(int)

args_dcsbm = (C_dcsbm, c, ℓ_dcsbm, θ_dcsbm, symmetric, make_connected)


generateSequence(
    outputfolder=output_folder_dcsbm, 
    model=DCSBM,             
    args=args_dcsbm,         
    n_graphs=20,             
    verbose=True             
)


ValueError: operands could not be broadcast together with shapes (4,4) (5,) 

ER

In [ ]:
k_er = 1
C_er = np.ones((k_er,k_er))*c
    
n_er= np.random.randint(int(n_nodes*(1-γ)), int(n_nodes*(1+γ)))
ℓ_er= np.zeros(n_er).astype(int)
θ_er = np.ones(n_er)
args_er = (C_er, c, ℓ_er, θ_er, symmetric, make_connected)


 
generateSequence(
    outputfolder=output_folder_er, 
    model=DCSBM,             
    args=args_er,         
    n_graphs=20,             
    verbose=True             
)

CM

In [33]:

def _sample_pareto(N, alpha, loc=0, scale=1, mean=None):
    """
    Sample N values from a pareto distribution
    """
    b = alpha-1
    r = pareto.rvs(b, loc=loc, scale=scale, size=N)
    if mean:
        r = (r/r.mean()*mean).round()
    return r

def getDegreeFromPareto(n, c, alpha):

    fitness_pareto = np.maximum(_sample_pareto(n, 2.5), 1e-12)
    fitness_pareto = fitness_pareto/np.mean(fitness_pareto)
    fitness_binomial = np.maximum(np.random.binomial(n, c/n, n)/c, 1e-12)
    fitness = np.exp(np.log(fitness_binomial)*(1-alpha) + np.log(fitness_pareto)*alpha)
    theta = fitness/np.mean(fitness)
    degree_sequence = np.random.binomial(n-1, np.minimum(1,theta/n*c))

    return degree_sequence

In [43]:

g=getDegreeFromPareto(100,5,2)
print("La somma attuale dei gradi è:", sum(g))

La somma attuale dei gradi è: 268


In [44]:
nx.configuration_model(g)

In [ ]:

n_conf = np.random.randint(int(n_nodes*(1-γ)), int(n_nodes*(1+γ)))
θ_conf = G = nx.configuration_model(degree_sequence)
θ_conf = θ_conf/np.mean(θ_conf)
ℓ_conf = np.zeros(n_conf).astype(int)
args_conf = (C_er, c, ℓ_conf, θ_conf, symmetric, make_connected)

generateSequence(
    outputfolder=output_folder_conf, 
    model=DCSBM,             
    args=args_conf,         
    n_graphs=20,             
    verbose=True             
)

GM

In [ ]:
β = 20
n_geo = np.random.randint(int(n_nodes*(1-γ)), int(n_nodes*(1+γ)))

r_geo = np.random.uniform(0, 1, n_geo)
θ_geo = np.random.uniform(0, 2*np.pi, n_geo)
X = np.zeros((n_geo, 2))
X[:,0] = r_geo*np.cos(θ_geo)
X[:,1] = r_geo*np.sin(θ_geo)
args_geo= (X, c_er, β)

generateSequence(
    outputfolder=output_folder_geo, 
    model= GeometricModel,             
    args=args_geo,         
    n_graphs=20,             
    verbose=True             
)
  

In [ ]:

def _sample_pareto(N, alpha, loc=0, scale=1, mean=None):
    """
    Sample N values from a pareto distribution
    """
    b = alpha-1
    r = pareto.rvs(b, loc=loc, scale=scale, size=N)
    if mean:
        r = (r/r.mean()*mean).round()
    return r

def getDegreeFromPareto(n, c, alpha):

    fitness_pareto = np.maximum(_sample_pareto(n, 2.5), 1e-12)
    fitness_pareto = fitness_pareto/np.mean(fitness_pareto)
    fitness_binomial = np.maximum(np.random.binomial(n, c/n, n)/c, 1e-12)
    fitness = np.exp(np.log(fitness_binomial)*(1-alpha) + np.log(fitness_pareto)*alpha)
    theta = fitness/np.mean(fitness)
    degree_sequence = np.random.binomial(n-1, np.minimum(1,theta/n*c))

    return degree_sequence